# 06 · Evaluación OCR Ecuador

Detector activo: **RT-DETRv2 R18** mediante **Transformers**, checkpoint `merve/license-plates-rtdetrv2` con licencia Apache-2.0. Los datos y métricas nuevos se mantienen separados de los resultados históricos YOLO.


In [ ]:
%pip install -q "transformers>=5.4,<6" "accelerate>=1.2,<2" "pycocotools>=2.0.8" "optuna>=4,<5" "fast-plate-ocr==1.1.0" "roboflow==1.7.1"

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO_ROOT=Path("/content/SIRAP_USA_ECUADOR")
if not REPO_ROOT.exists(): subprocess.run(["git","clone","--depth","1","https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git",str(REPO_ROOT)],check=True)
os.chdir(REPO_ROOT); sys.path.insert(0,str(REPO_ROOT))
COUNTRY="ecuador"
MODEL_ID="merve/license-plates-rtdetrv2"
print("Perfil:",COUNTRY,"| RT-DETRv2 R18:",MODEL_ID)


## OCR independiente del detector
El cambio de detector no modifica fast-plate-ocr. Evalúa recortes de placas con referencia conocida y registra el modelo OCR, dataset y fecha. No combines estas mediciones con el mAP del detector.

In [ ]:
import pandas as pd
from pathlib import Path
from src.data_processing import load_image,normalize_plate
from src.model import PlateReader
CSV_PATH=None  # CSV con image_path y plate_gt
if CSV_PATH:
    reader=PlateReader(); data=pd.read_csv(CSV_PATH); rows=[]
    for row in data.itertuples(index=False):
        raw,conf,mean=reader.read(load_image(Path(row.image_path).read_bytes()))
        rows.append({"plate_gt":normalize_plate(row.plate_gt),"plate_pred":normalize_plate(raw),
                     "ocr_confidence":conf,"ocr_mean_confidence":mean})
    display(pd.DataFrame(rows).head())
else: print("Asigna CSV_PATH para evaluar recortes etiquetados.")
